# SectorCycle 경기 국면 판별기 — 빈칸 연습

**원본**: `notebooks/SectorCycle.ipynb`

`___` 빈칸을 채워서 코드를 완성하세요.

총 **84개** 빈칸이 있습니다.

In [1]:
# ── Cell 0: 한글 폰트 설정 ─────────────────────────────────────────────────
import matplotlib.pyplot as plt            # matplotlib 차트 라이브러리 임포트
import matplotlib.font_manager as fm       # 폰트 관리 모듈 임포트

# 한글 폰트 파일 경로 지정 (Windows 기본 맑은고딕)
font_path = 'C:/Windows/Fonts/malgun.ttf'
# 폰트 파일에서 폰트 이름을 추출 (예: 'Malgun Gothic')
font_name = fm.FontProperties(fname=font_path).get_name()
# matplotlib 전역 폰트를 한글 폰트로 설정 (차트 한글 깨짐 방지)
plt.rc('font', family=font_name)

## Q1. 라이브러리 임포트 (4 blanks)
- HMM 클래스명은? `GaussianHMM` / `MultinomialHMM`
- 스케일러 클래스명은? 평균=0, 분산=1로 변환


In [2]:
# ── Cell 1: 라이브러리 로드 및 전역 설정 ───────────────────────────────────

import warnings                            # 경고 메시지 제어 모듈
warnings.filterwarnings('ignore')          # 모든 경고 무시 (deprecated 등 억제)

# ── 데이터 처리 및 시각화 라이브러리 ──
import pandas as pd                        # 데이터프레임 생성·조작·병합
import numpy as np                         # 배열 연산, 수학 함수
import matplotlib.pyplot as plt            # 기본 차트 (막대, 선, 산점도 등)
import seaborn as sns                      # 고급 통계 시각화 (히트맵, 상관행렬 등)
import yfinance as yf                      # Yahoo Finance API로 주가 데이터 다운로드

# ── 머신러닝 라이브러리 ──
from hmmlearn.hmm import GaussianHMM               # Q1-1: 가우시안 HMM 클래스 (비지도 학습 기반 국면 분류)
from sklearn.preprocessing import StandardScaler      # Q1-2: 피처 표준화 스케일러 (평균=0, 분산=1)

## Q2. 경기 국면 상수 정의 (6 blanks)
- `RANK_TO_PHASE`: HMM 복합점수 낮은 순 → 국면 매핑
  - rank 0 (최저점수) → 침체(3), rank 1 → 둔화(2), rank 2 → 회복(0), rank 3 → 확장(1)
- `PHASE_NAMES`: 0=회복, 1=확장, 2=둔화, 3=침체


In [3]:
# ── 경기 국면 4단계 정의 ──
# 0=회복(경기 바닥→상승), 1=확장(호황), 2=둔화(고점→하락), 3=침체(불황)
PHASE_NAMES  = {0: '🌱 회복', 1: '☀️ 확장', 2: '🍂 둔화', 3: '❄️ 침체'}  # 국면 이름 딕셔너리
# 각 국면별 차트 색상 (녹=회복, 노랑=확장, 주황=둔화, 파랑=침체)
PHASE_COLORS = {0: '#4CAF50', 1: '#FFC107', 2: '#FF9800', 3: '#2196F3'}  # 국면별 색상 코드

# HMM 상태 정렬 후 국면 매핑: rank 0(최저)→침체, 1→둔화, 2→회복, 3→확장
RANK_TO_PHASE = [3, 2, 0, 1]       # Q2-1~4: 침체=3, 둔화=2, 회복=0, 확장=1 순서로 매핑

# ── 분석 대상 종목 설정 ──
# S&P 500 섹터별 SPDR ETF + SOXX(반도체)
SECTOR_ETFS  = ['XLF', 'XLE', 'XLK', 'XLV', 'XLB', 'XLP', 'XLU', 'XLI', 'XLRE', 'SOXX']
# 사용자 보유 종목
MY_HOLDINGS  = ['QQQ', 'SPY']              # 보유 종목 리스트

# 설정 확인 출력
print('라이브러리 로드 완료')
print(f'섹터 ETF: {SECTOR_ETFS}')
print(f'보유 종목: {MY_HOLDINGS}')

라이브러리 로드 완료
섹터 ETF: ['XLF', 'XLE', 'XLK', 'XLV', 'XLB', 'XLP', 'XLU', 'XLI', 'XLRE', 'SOXX']
보유 종목: ['QQQ', 'SPY']


## Q3. FRED 데이터 다운로드 함수 (6 blanks)
- CSV를 DataFrame으로 파싱할 때: `index_col`, `parse_dates` 옵션
- 숫자 변환 실패 시 NaN 처리: `errors='coerce'`
- 지수 백오프 공식: `2 ** attempt`


In [4]:
# ── Cell 2: FRED 매크로 경제지표 다운로드 ──────────────────────────────────
import requests, time                      # requests: HTTP 요청 / time: 재시도 대기
from io import StringIO                    # 문자열을 파일 객체처럼 변환 (pd.read_csv에 전달)

# FRED 무료 CSV 다운로드 기본 URL (시리즈 ID를 뒤에 붙여서 사용)
FRED_BASE = 'https://fred.stlouisfed.org/graph/fredgraph.csv?id='
# 브라우저처럼 보이는 User-Agent 헤더 (FRED가 Python 기본 UA를 차단할 수 있음)
HEADERS   = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

def fetch_fred(series_id, col_name, retries=4, timeout=30):
    """FRED에서 시계열 CSV를 다운로드하고 DataFrame으로 반환.
    실패 시 지수 백오프(1→2→4→8초)로 최대 4회 재시도."""
    url = FRED_BASE + series_id            # 다운로드할 전체 URL 생성
    for attempt in range(retries):         # 최대 retries번 시도
        try:
            resp = requests.get(url, headers=HEADERS, timeout=timeout)  # HTTP GET 요청
            resp.raise_for_status()                     # Q3-1: HTTP 에러(4xx/5xx) 발생 시 예외 던지는 메소드
            df = pd.read_csv(              # CSV 텍스트를 DataFrame으로 파싱
                StringIO(resp.text),       # 응답 텍스트를 파일 객체로 감싸기
                index_col= 0,            # Q3-2: 첫 번째 열(날짜)을 인덱스로 사용하는 값
                parse_dates= True            # Q3-3: 인덱스를 datetime으로 자동 변환 (True/False)
            )
            df.columns = [col_name]        # 컬럼명을 사용자 지정 이름으로 변경
            df[col_name] = pd.to_numeric(  # 값을 숫자로 변환
                df[col_name], errors='coerce'  # Q3-4: 변환 실패 시 NaN 처리하는 옵션값
            )
            return df                      # 성공 시 DataFrame 반환
        except Exception as e:             # 다운로드 실패 시
            if attempt < retries - 1:      # 마지막 시도가 아니면
                wait = 2 ** attempt      # Q3-5: 지수 백오프 밑(base) 값
                print(f'  [{series_id}] 재시도 {attempt+1}/{retries} ({wait}초 대기)...')
                time.sleep(wait)             # Q3-6: 대기 함수
            else:
                raise                      # 마지막 시도도 실패하면 예외를 호출자에게 전파

In [6]:
print('FRED 데이터 다운로드 중...')

# ── 8개 FRED 시리즈 다운로드 ──
# 1) 산업생산지수 (월별, 지수) → PMI 유사 지수로 변환 예정
df_indpro   = fetch_fred('INDPRO',  'indpro')
print('  ✓ INDPRO (산업생산지수)')

# 2) 10년-3개월 국채 금리차 (%, 일별→월별 리샘플링)
df_yield    = fetch_fred('T10Y3M',  'yield_spread')
print('  ✓ T10Y3M (장단기 금리차)')

# 3) 시카고 연준 국가 금융환경지수 (주별, 지수)
#    양수=긴축적 / 음수=완화적 / 0=평균적 금융환경
df_anfci    = fetch_fred('ANFCI',   'anfci')
print('  ✓ ANFCI (금융환경지수)')

# 4) 신규 실업급여 청구건수 (주별, 천 명)
#    고용시장 선행지표: 증가=해고 증가=경기 악화 신호
df_icsa     = fetch_fred('ICSA',    'icsa')
print('  ✓ ICSA (신규 실업급여)')

# 5) 신규 건축 허가건수 (월별, 천 호)
#    주택시장 선행지표: 증가=건설 투자 증가=경기 회복 신호
df_permit   = fetch_fred('PERMIT',  'permit')
print('  ✓ PERMIT (건축허가)')

# 6) 실질 소매판매 (월별, 백만 달러, 인플레이션 조정)
#    소비 동행지표: 물가 영향 제거한 실질 소비 수준
df_rrsfs    = fetch_fred('RRSFS',   'real_retail')
print('  ✓ RRSFS (실질 소매판매)')

# 7) 비방위 자본재 신규주문 - 항공기 제외 (월별, 백만 달러)
#    기업 투자 선행지표: 설비투자 의향 반영
df_capex    = fetch_fred('ANDENO',  'capex_orders')
print('  ✓ ANDENO (자본재 주문)')

# 8) 실질 개인소득 (이전지출 제외, 월별, 십억 달러)
#    소득 동행지표: 실질 구매력 수준
df_w875rx1  = fetch_fred('W875RX1', 'real_income')
print('  ✓ W875RX1 (실질 개인소득)')

FRED 데이터 다운로드 중...
  ✓ INDPRO (산업생산지수)
  ✓ T10Y3M (장단기 금리차)
  ✓ ANFCI (금융환경지수)
  ✓ ICSA (신규 실업급여)
  ✓ PERMIT (건축허가)
  ✓ RRSFS (실질 소매판매)
  ✓ ANDENO (자본재 주문)
  ✓ W875RX1 (실질 개인소득)


## Q4. INDPRO → PMI 유사 지수 변환 (7 blanks)
- 몇 개월 변화율(%)로 모멘텀 산출? → `pct_change(?)`
- 롤링 윈도우 크기는? (10년 = ?개월)
- 최소 관측치 수는? (3년 = ?개월)
- z-score → PMI 스케일 변환: `z * ? + ?`
- 클리핑 범위: `np.clip(값, ?, ?)`


In [ ]:
# ── INDPRO → PMI 유사 지수 변환 ──
# INDPRO 원본은 절대 지수(예: 103.5)이므로 3개월 변화율(%)로 모멘텀 산출
indpro_mom = df_indpro['indpro'].___(___)  * 100  # Q4-1~2: 변화율 메소드와 기간 (3개월)
# 120개월(10년) rolling 윈도우에서 z-score 계산:
#   z = (현재값 - 10년 평균) / 10년 표준편차
#   → z * 10 + 50 으로 변환하면 평균=50, 표준편차=10인 PMI 유사 스케일
pmi_series = indpro_mom.rolling(___, min_periods=___).apply(  # Q4-3~4: 롤링 윈도우(120), 최소 관측치(36)
    lambda x: np.clip(
        (x.iloc[-1] - x.___()) / max(x.___(), 0.01) * ___ + ___,  # Q4-5~6: 평균메소드, 표준편차메소드 / Q4-7~8: 스케일(10, 50)
        10, 90                             # PMI 범위를 10~90으로 클리핑
    ),
    raw=False                              # Series 형태로 전달 (iloc 사용 가능)
)
# PMI 시리즈를 DataFrame으로 변환 (NaN 제거)
df_pmi = pmi_series.rename('pmi').___().to_frame()  # Q4-9: 결측값 제거 메소드

In [10]:
# indpro 원본은 절대지수 이므로 3개월 변화율 모멘텀 산출
indpro_mom = df_indpro['indpro'].pct_change(3) * 100 

# 120개월 rolling 윈도우에서 z-score 계산
pmi_series = indpro_mom.rolling(120, min_periods=36).apply(
    lambda x: np.clip(
        (x.iloc[-1] - x.mean())/max(x.std(),0.01) * 10 + 50,
        10, 90
    ),
    raw = False)
# PMI 시리즈를 dataframe으로 변환
df_pmi = pmi_series.rename('pmi').dropna().to_frame()

## Q5. 파생 변수 변환 — ICSA, ANFCI 리샘플링 (6 blanks)
- 주별 데이터 → 월별 평균: `.resample('MS').?()`
- 전년동월비(YoY%): `.pct_change(?) * 100` — 기간은 12개월
- 결측값 제거 후 필요한 열만 선택


In [ ]:
# ── 파생 변수 변환 (1/2): ICSA, ANFCI ──

# ICSA (주별) → 월별 평균으로 리샘플링 후 YoY% 변환
df_icsa = df_icsa.resample('MS').___()                       # Q5-1: 주별→월별 평균 집계 메소드
df_icsa['icsa_yoy'] = df_icsa['icsa'].___(___) * 100         # Q5-2~3: 변화율 메소드와 YoY 기간(12)
df_icsa = df_icsa[['icsa_yoy']].___()                        # Q5-4: 결측값 제거 메소드

# ANFCI (주별) → 월별 평균으로 리샘플링 (원본 지수 그대로 사용)
df_anfci = df_anfci.resample('MS').___()                     # Q5-5: 주별→월별 평균 집계 메소드

# PERMIT → YoY% 변환 (건축허가 전년동월비)
df_permit['permit_yoy'] = df_permit['permit'].pct_change(___) * 100  # Q5-6: YoY 기간(12)
df_permit = df_permit[['permit_yoy']].dropna()               # 결측값 제거 후 필요 열만 선택

In [ ]:
df_ics = df_icsa.resample('MS').mean()
df_icsa['icsa_yoy'] = df_icsa['icsa'].pct_change(12) * 100 # icsa 변화율
df_icsa = df_icsa[['icsa_yoy']].dropna()

df_anfci = df_anfci.resample('MS').mean() #월별 평균 리샘플링

df_permit['permit_yoy'] = df_permit['permit'].pct_change(12) * 100
df_permit = df_permit[['permit_yoy']].dropna()

# 각 경제 지표를 월별 기준으로 정규화

## Q6. 파생 변수 변환 — 소매판매, 자본재, 소득 (6 blanks)
- 동일한 YoY% 패턴: `.pct_change(?) * 100`
- 각각 `real_retail`, `capex_orders`, `real_income` 원본 컬럼에서 변환


In [ ]:
# ── 파생 변수 변환 (2/2): 소매판매, 자본재, 소득 ──

# RRSFS (실질 소매판매) → YoY% 변환
df_rrsfs['real_retail_yoy'] = df_rrsfs['real_retail'].___(___) * 100  # Q6-1~2: 변화율 메소드와 기간
df_rrsfs = df_rrsfs[['real_retail_yoy']].___()               # Q6-3: 결측값 제거 메소드

# ANDENO (자본재 주문) → YoY% 변환
df_capex['capex_yoy'] = df_capex['capex_orders'].pct_change(___) * 100  # Q6-4: YoY 기간
df_capex = df_capex[['capex_yoy']].dropna()                 # 결측값 제거

# W875RX1 (실질 개인소득) → YoY% 변환
df_w875rx1['real_income_yoy'] = df_w875rx1['real_income'].pct_change(___) * 100  # Q6-5: YoY 기간
df_w875rx1 = df_w875rx1[['real_income_yoy']].___()           # Q6-6: 결측값 제거 메소드

## Q7. 매크로 지표 병합 (4 blanks)
- 8개 DataFrame을 `.join()`으로 병합
- 월 시작일 기준 리샘플링: `.resample('MS').?()`
- join 방식은? `inner` / `outer` / `left`


In [ ]:
# ── 8개 매크로 지표를 하나의 DataFrame으로 병합 ──
macro = (df_pmi                              # PMI 유사 지수 (기준 DataFrame)
         .join(df_yield,    how='___')        # Q7-1: 병합 방식 (모든 날짜 포함)
         .join(df_anfci,    how='outer')      # + 금융환경지수
         .join(df_icsa,     how='outer')      # + 신규 실업급여 YoY%
         .join(df_permit,   how='outer')      # + 건축허가 YoY%
         .join(df_rrsfs,    how='outer')      # + 실질 소매판매 YoY%
         .join(df_capex,    how='outer')      # + 자본재 주문 YoY%
         .join(df_w875rx1,  how='outer'))     # + 실질 개인소득 YoY%
macro = macro.resample('MS').___()            # Q7-2: 월 시작일 기준 마지막 값 선택 메소드
macro = macro.___()                           # Q7-3: 결측값이 있는 행 제거 메소드

# 결과 확인 출력
print(f'\n매크로 데이터: {macro.shape} | {macro.index[0].date()} ~ {macro.index[-1].date()}')
print(f'변수: {list(macro.columns)}')
macro.tail()

FRED 에서 매크로 데이터 지표 다운로드

## Q8. 섹터 ETF 다운로드 및 월별 수익률 (4 blanks)
- 타임존 제거: `.tz_localize(?)`
- 월별 리샘플링 후 마지막 값: `.resample('MS').?()`
- 월별 수익률 계산: `.?()`  (전월 대비 변화율)


In [ ]:
# ── Cell 3: 섹터 ETF + 보유 종목 월별 수익률 다운로드 ─────────────────────
# 섹터 ETF + 보유 종목을 하나의 리스트로 합침
ALL_TICKERS = SECTOR_ETFS + MY_HOLDINGS    # 전체 종목 리스트 생성

# 섹터 ETF(XLF 등)는 1998-12-22 상장 → 그 이전 데이터는 존재하지 않음
ETF_START   = '1999-01-01'                 # ETF 상장 이후 시작일
# 매크로 데이터 시작일과 ETF 상장일 중 더 늦은 날짜를 시작일로 사용
START_DATE  = max(str(macro.index[0].date()), ETF_START)  # 두 날짜 중 늦은 날짜 선택

print(f'{len(ALL_TICKERS)}개 종목 다운로드 중... (시작: {START_DATE})')

# yf.download()는 일괄 다운로드 시 간헐적 실패가 많음
# → yf.Ticker().history()로 종목 하나씩 개별 다운로드 (더 안정적)
frames = {}                                # 종목별 종가를 담을 딕셔너리
for ticker in ALL_TICKERS:                 # 각 종목에 대해 반복
    try:
        t = yf.Ticker(ticker)              # Yahoo Finance 종목 객체 생성
        hist = t.history(                  # 과거 주가 데이터 요청
            start=START_DATE,              # 시작일 지정
            auto_adjust=True               # 수정주가(배당·분할 반영) 사용
        )
        if len(hist) > 0:                  # 데이터가 존재하면
            frames[ticker] = hist['Close'] # 종가(Close) 컬럼만 저장
            print(f'  ✓ {ticker}: {len(hist)}일')  # 성공 로그 출력
        else:
            print(f'  ✗ {ticker}: 데이터 없음')     # 빈 데이터 로그
    except Exception as e:
        print(f'  ✗ {ticker}: {e}')        # 에러 발생 시 메시지 출력

In [ ]:
# 딕셔너리 → DataFrame 변환 (열=종목, 행=날짜)
raw = pd.DataFrame(frames)                 # 종목별 종가 딕셔너리를 DataFrame으로 변환
# 인덱스를 DatetimeIndex로 명시적 변환 (문자열 인덱스 방지)
raw.index = pd.to_datetime(raw.index)      # 인덱스를 날짜 타입으로 변환

# yfinance 1.2+는 tz-aware 인덱스(예: -05:00) 반환
# FRED 데이터는 tz-naive(타임존 없음)이므로 join 시 에러 발생
# → 타임존 정보를 제거하여 형식 통일
if raw.index.tz is not None:               # 타임존 정보가 있으면
    raw.index = raw.index.tz_localize(___) # Q8-1: 타임존 제거 인자 (tz-aware → tz-naive)

# 일별 종가 → 월별 종가: 각 월의 마지막 거래일 종가 사용
monthly_prices  = raw.resample('MS').___()  # Q8-2: 월말 마지막 값 선택 메소드
# 월별 수익률 계산: (이번 달 종가 - 지난 달 종가) / 지난 달 종가
monthly_returns = monthly_prices.___()      # Q8-3: 전월 대비 변화율 계산 메소드

# 전체 수익률에서 섹터 ETF와 보유 종목을 분리 (다운로드 실패한 종목은 자동 제외)
sector_ret  = monthly_returns[[c for c in SECTOR_ETFS if c in monthly_returns.columns]]  # 섹터 ETF만 추출
holding_ret = monthly_returns[[c for c in MY_HOLDINGS if c in monthly_returns.columns]]  # 보유 종목만 추출

# 결과 확인 출력
print(f'\n섹터 ETF 수익률: {sector_ret.shape}')   # (행=개월수, 열=ETF수)
print(f'보유 종목 수익률: {holding_ret.shape}')
sector_ret.tail(3)                         # 최근 3개월 수익률 미리보기

## Q9. 파생 피처 생성 및 데이터 병합 (5 blanks)
- PMI 3개월 변화량: `.diff(?)`
- 자본재 주문 YoY% 3개월 변화량: `.diff(?)`
- DataFrame 병합: `.join(df, how='?')`


In [ ]:
# ── Cell 4: 파생 피처 생성 및 데이터 병합 ──────────────────────────────────
# 매크로 지표 원본을 복사 (원본 보존)
feat = macro.copy()                        # 원본 DataFrame을 깊은 복사
# PMI 3개월 변화량: 3개월 전 PMI와의 차이 → 경기 모멘텀 방향 (양수=개선, 음수=악화)
feat['pmi_chg3m']     = feat['pmi'].___(___) # Q9-1~2: 차분 메소드와 기간 (3개월)
# 자본재 주문 YoY% 3개월 변화량 → 기업 투자 모멘텀 가속/감속
feat['capex_yoy_chg3m'] = feat['capex_yoy'].___(___) # Q9-3~4: 차분 메소드와 기간 (3개월)

# 매크로 피처(feat) + 섹터 ETF 수익률(sector_ret)을 날짜 기준으로 병합
df = feat.join(sector_ret,  how='___')     # Q9-5: 병합 방식 (왼쪽 기준)
# 보유 종목 수익률(holding_ret)도 같은 방식으로 병합
df = df.join(holding_ret,   how='left')    # 보유 종목 수익률 병합
# 결측값이 있는 행 제거 (diff로 인한 첫 3행, ETF 상장 전 기간 등)
df = df.dropna()                           # 모든 NaN 행 제거

In [ ]:
# 모델 학습에 사용할 10개 피처 목록 정의
# pmi: 산업생산 기반 경기 지수 (50 중심, 동행)
# yield_spread: 10년-3개월 금리차 (음수=역전=침체 신호, 선행)
# anfci: 금융환경지수 (양수=긴축, 음수=완화, M2 대체)
# icsa_yoy: 신규 실업급여 전년비 (고용 선행)
# permit_yoy: 건축허가 전년비 (주택 선행)
# real_retail_yoy: 실질 소매판매 전년비 (소비 동행)
# capex_yoy: 자본재 주문 전년비 (투자 선행)
# real_income_yoy: 실질 개인소득 전년비 (소득 동행)
# pmi_chg3m: PMI 3개월 모멘텀 (경기 방향)
# capex_yoy_chg3m: 자본재 주문 YoY 3개월 변화 (투자 가속도)
FEATURE_COLS = ['pmi', 'yield_spread', 'anfci', 'icsa_yoy',
                'permit_yoy', 'real_retail_yoy', 'capex_yoy',
                'real_income_yoy', 'pmi_chg3m', 'capex_yoy_chg3m']

# 결과 확인 출력
print(f'결합 데이터: {df.shape} | {df.index[0].date()} ~ {df.index[-1].date()}')
print(f'피처 {len(FEATURE_COLS)}개: {FEATURE_COLS}')
df[FEATURE_COLS].describe()

매크로 피처 + 섹터 ETF 수익률 + 보유 종목 수익률

학습 피처 정의

## Q10. 국면 매핑 함수 (4 blanks)
- 복합점수: PMI 수준 + ? × PMI 3개월 모멘텀 (가중치=0.5)
- 복합점수를 낮은 순으로 정렬하는 numpy 함수는?
- `model.means_`에서 피처별 평균값 추출


In [ ]:
# ── Cell 5: HMM 학습 + 경기 국면 매핑 + 시각화 ────────────────────────────

def map_states_to_phases(model, pmi_idx=0, pmi_chg_idx=8):
    """HMM hidden states → 경기국면 매핑 (PMI 수준 + 모멘텀 복합점수 기준).
    
    PMI가 높고 모멘텀이 양수인 상태 = 확장,
    PMI가 낮은 상태 = 침체, 중간이면서 모멘텀으로 회복/둔화 구분.
    
    Returns: {hmm_state_id: phase_id} 예: {2: 0, 0: 1, 3: 2, 1: 3}
    """
    # 복합점수: PMI 수준 + 0.5 × PMI 3개월 모멘텀
    scores = model.___[:, pmi_idx] + ___ * model.___[:, pmi_chg_idx]  # Q10-1~3: 모델 평균 속성(means_), 모멘텀 가중치(0.5), 모델 평균 속성(means_)
    sorted_states = np.___(scores)         # Q10-4: 낮은 순으로 인덱스 정렬하는 함수
    return {int(sid): RANK_TO_PHASE[rank] for rank, sid in enumerate(sorted_states)}  # 정렬된 rank → 국면 매핑

## Q11. 피처 스케일링 (3 blanks)
- 스케일러 클래스 생성: `?()`
- 학습 + 변환을 동시에: `.?(X)` — fit과 transform을 한 번에


In [ ]:
# ── 피처 스케일링 ──
X = df[FEATURE_COLS].values                # 학습 피처를 numpy 배열로 변환
scaler = ___()                             # Q11-1: 표준화 스케일러 객체 생성 (평균=0, 분산=1)
X_scaled = scaler.___(X)                   # Q11-2: 학습 데이터에 맞춰 스케일링 적용 (fit + transform)

## Q12. GaussianHMM 학습 (7 blanks)
- 경기국면 수(n_components)는?
- 공분산 타입: 먼저 `full` 시도 → 실패 시 `diag` 폴백
- EM 알고리즘 최대 반복 횟수: `n_iter=?`
- 재현성 보장: `random_state=?`
- 학습 메소드: `.?(X_scaled)`


In [ ]:
# ── GaussianHMM 학습 (4개 상태 = 4개 경기국면) ──
# full 공분산 시도 → 실패 시 diag 폴백 (feature1_regime.py와 동일 패턴)
for cov_type in ('___', '___'):            # Q12-1~2: 공분산 타입 2가지 (full 먼저, diag 폴백)
    try:
        model = GaussianHMM(
            n_components=___,              # Q12-3: 경기국면 수 (회복/확장/둔화/침체)
            covariance_type=cov_type,      # 공분산 타입 (full: 피처 간 상관관계 학습)
            n_iter=___,                    # Q12-4: EM 알고리즘 최대 반복 횟수
            random_state=___,              # Q12-5: 재현성 보장 시드값
        )
        model.___(X_scaled)                # Q12-6: EM 알고리즘으로 파라미터 추정하는 학습 메소드
        print(f'HMM 학습 완료 (covariance: {cov_type})')
        break                              # 학습 성공 시 루프 탈출
    except (ValueError, np.linalg.LinAlgError):  # 수렴 실패 또는 행렬 오류 시
        if cov_type == 'diag':             # diag도 실패하면
            raise                          # 예외를 상위로 전파

## Q13. 상태 예측 및 국면 매핑 (3 blanks)
- Viterbi 알고리즘으로 최적 상태 시퀀스 추론: `model.?(X_scaled)`
- 모델 로그우도 점수: `model.?(X_scaled)`


In [ ]:
# ── 상태 예측 및 국면 매핑 ──
states = model.___(X_scaled)               # Q13-1: Viterbi로 최적 상태 시퀀스 추론하는 메소드
state_to_phase = map_states_to_phases(model)  # HMM 상태 → 경기국면 매핑 딕셔너리 생성
df['phase'] = [state_to_phase[s] for s in states]  # 각 시점의 상태를 국면으로 변환하여 컬럼 추가

# 매핑 결과 출력
print('\nHMM 상태 → 경기국면 매핑:')
for s in range(4):                         # 4개 상태에 대해 반복
    ph = state_to_phase[s]                 # 상태에 대응하는 국면 번호
    score = model.means_[s, 0] + 0.5 * model.means_[s, 8]  # 복합점수 계산
    print(f'  state {s} → {PHASE_NAMES[ph]}  (복합점수: {score:.4f})')

print(f'\nLog-likelihood: {model.___(X_scaled):.2f}')  # Q13-2: 로그우도 점수 메소드
print(f'Per-sample LL: {model.___(X_scaled)/len(X_scaled):.4f}')  # Q13-3: 로그우도 점수 메소드

In [ ]:
# ── 시각화: 국면 분포 ──
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# [좌측] 국면별 관측 횟수 막대 그래프
counts = df['phase'].value_counts().sort_index()
colors = [PHASE_COLORS[i] for i in counts.index]
axes[0].bar(
    [PHASE_NAMES[i] for i in counts.index],
    counts.values,
    color=colors
)
axes[0].set_title('국면별 관측 횟수 (HMM)')
axes[0].set_ylabel('개월 수')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontsize=11)

# [우측] PMI 시계열 산점도 — 색상이 HMM 예측 국면
axes[1].scatter(
    df.index, df['pmi'],
    c=[PHASE_COLORS[p] for p in df['phase']],
    s=20, alpha=0.8
)
axes[1].axhline(50, color='gray', linestyle='--', alpha=0.5, label='PMI=50')
axes[1].set_title('PMI 시계열 (색상=HMM 국면)')
axes[1].set_ylabel('PMI')
axes[1].legend()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=PHASE_COLORS[k], label=PHASE_NAMES[k]) for k in range(4)]
fig.legend(handles=legend_elements, loc='upper center', ncol=4, bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
plt.show()

# 국면별 분포 텍스트 출력
print('\n국면 분포:')
for ph, cnt in counts.items():
    print(f'  {PHASE_NAMES[ph]}: {cnt}개월 ({cnt/len(df)*100:.1f}%)')

## Q14. 전이확률 행렬 및 모델 평가 (5 blanks)
- 전이확률 행렬 속성: `model.?`
- 평균 지속 기간 공식: `? / (? - 자기전이확률)`
- 로그우도 점수: `model.?(X_scaled)`


In [ ]:
# ── Cell 6: HMM 모델 평가 ──────────────────────────────────────────────
# HMM은 비지도학습이므로 train/test 분할 없이 전체 데이터로 학습
# 대신 log-likelihood와 전이확률 행렬로 모델 품질 평가

# ── 전이 확률 행렬 시각화 ──
trans_mat = model.___                      # Q14-1: 전이확률 행렬 속성
phase_labels = [PHASE_NAMES[state_to_phase[s]] for s in range(4)]  # 각 상태의 국면 이름 리스트

# 국면 순서로 재배열 (회복, 확장, 둔화, 침체 순)
phase_order = []                           # 국면 순서에 해당하는 HMM 상태 번호 리스트
for ph in range(4):                        # 회복(0), 확장(1), 둔화(2), 침체(3) 순서로
    for s in range(4):                     # 4개 HMM 상태 중
        if state_to_phase[s] == ph:        # 해당 국면에 매핑된 상태를 찾아
            phase_order.append(s)          # 순서 리스트에 추가
            break                          # 찾으면 다음 국면으로

# 전이확률 행렬을 국면 순서로 재배열 (행과 열 모두)
reordered = trans_mat[np.ix_(phase_order, phase_order)]  # 행·열을 국면 순서로 인덱싱
reordered_labels = [PHASE_NAMES[state_to_phase[s]] for s in phase_order]  # 재배열된 라벨

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# [좌측] 전이 확률 히트맵
sns.heatmap(
    reordered, annot=True, fmt='.2f',
    cmap='YlOrRd',
    xticklabels=reordered_labels,
    yticklabels=reordered_labels,
    ax=axes[0],
    vmin=0, vmax=1,
    cbar_kws={'label': '전이 확률'}
)
axes[0].set_title('경기국면 전이 확률 행렬')
axes[0].set_xlabel('다음 국면')
axes[0].set_ylabel('현재 국면')

# [우측] 국면별 샘플 수 + 평균 지속 기간
x = np.arange(4)                           # x축 위치 배열
counts_ordered = [int((df['phase'] == ph).sum()) for ph in range(4)]  # 국면별 관측 횟수
# 평균 지속 기간 = 1 / (1 - 자기전이확률)
durations = [___ / (___ - reordered[i, i]) if reordered[i, i] < 1 else float('inf') for i in range(4)]  # Q14-2~3: 분자(1), 분모에서 뺄 값(1)

bars = axes[1].bar(
    [PHASE_NAMES[ph] for ph in range(4)],
    counts_ordered,
    color=[PHASE_COLORS[ph] for ph in range(4)],
    alpha=0.8
)
axes[1].set_title('국면별 관측 횟수 및 평균 지속기간')
axes[1].set_ylabel('개월 수')
for i, (v, d) in enumerate(zip(counts_ordered, durations)):
    axes[1].text(i, v + 0.5, f'{v}개월\n(평균 {d:.0f}개월)', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# 모델 성능 요약
avg_ll = model.___(X_scaled) / len(X_scaled)  # Q14-4: 로그우도 점수 메소드
print(f'Per-sample Log-likelihood: {avg_ll:.4f}')
print(f'전체 데이터: {len(X_scaled)}개월')
print(f'\n전이 확률 요약:')
for i, ph in enumerate(range(4)):          # 4개 국면에 대해 반복
    s = phase_order[i]                     # 국면에 해당하는 HMM 상태 번호
    self_prob = trans_mat[s, s]            # 자기전이확률 (현재 국면 유지 확률)
    print(f'  {PHASE_NAMES[ph]}: 자기전이 {self_prob:.1%} → 평균 지속 {___/(___-self_prob):.1f}개월')  # Q14-5: 평균 지속 기간 공식 (1/(1-p))

## Q15. HMM 발산 분포 분석 (3 blanks)
- 스케일링된 emission means를 원본 스케일로 되돌리는 메소드?
- `model.means_`에서 국면 순서로 인덱싱


In [ ]:
# ── Cell 7: HMM 발산 분포 분석 (국면별 매크로 지표 특성) ───────────────────
# HMM 각 상태의 emission mean을 국면별로 시각화 → 국면이 경제적으로 의미 있는지 확인

# 국면 순서로 emission means 재배열
means_reordered = scaler.___(model.___[phase_order])  # Q15-1~2: 역변환 메소드, 모델 평균 속성
feature_labels = ['PMI', '금리차', 'ANFCI', '실업급여\nYoY', '건축허가\nYoY',
                  '소매판매\nYoY', '자본재\nYoY', '실질소득\nYoY', 'PMI\n3개월변화', '자본재\n3개월변화']

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# [상단] 국면별 피처 평균값 그룹 막대 그래프
x = np.arange(len(FEATURE_COLS))
width = 0.2
for i, ph in enumerate(range(4)):
    axes[0].bar(
        x + i * width - 1.5 * width,
        means_reordered[i],
        width,
        label=PHASE_NAMES[ph],
        color=PHASE_COLORS[ph],
        alpha=0.85
    )
axes[0].set_xticks(x)
axes[0].set_xticklabels(feature_labels, fontsize=9)
axes[0].set_title('국면별 매크로 지표 평균값 (HMM Emission Means, 원본 스케일)')
axes[0].legend()
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].grid(axis='y', alpha=0.3)

# [하단] 주요 피처 3개의 국면별 분포 (박스플롯)
key_features = ['pmi', 'yield_spread', 'pmi_chg3m']
key_labels = ['PMI', '장단기 금리차', 'PMI 3개월 변화']

for idx, (feat, label) in enumerate(zip(key_features, key_labels)):
    ax_pos = idx
    for ph in range(4):
        mask = df['phase'] == ph
        vals = df.loc[mask, feat].values
        pos = idx * 5 + ph
        bp = axes[1].boxplot(
            vals, positions=[pos], widths=0.6,
            patch_artist=True,
            boxprops=dict(facecolor=PHASE_COLORS[ph], alpha=0.7),
            medianprops=dict(color='black'),
            showfliers=False
        )
    if idx < 2:
        axes[1].axvline(idx * 5 + 3.5, color='gray', linewidth=0.5, linestyle='--')

axes[1].set_xticks([1.5, 6.5, 11.5])
axes[1].set_xticklabels(key_labels)
axes[1].set_title('주요 피처의 국면별 분포')
axes[1].set_ylabel('값')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=PHASE_COLORS[k], label=PHASE_NAMES[k]) for k in range(4)]
axes[1].legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

# 국면별 매크로 특성 요약
print('\n국면별 주요 지표 평균:')
print(f'{"":12s} {"PMI":>8s} {"금리차":>8s} {"ANFCI":>8s} {"PMI모멘텀":>10s}')
print('-' * 50)
for i, ph in enumerate(range(4)):
    m = means_reordered[i]
    print(f'{PHASE_NAMES[ph]:12s} {m[0]:8.1f} {m[1]:8.2f} {m[2]:8.2f} {m[___]:10.2f}')  # Q15-3: PMI 3개월변화 인덱스 (0부터 시작)

## Q16. 국면별 섹터 성과 분석 (6 blanks)
- 그룹별 평균: `.groupby('phase')[cols].?()`
- 소수 → 퍼센트 변환: `* ?`
- 확률 예측 메소드: `model.?(X_scaled)`
- 상위 N개 선택: `.?(3)`


In [ ]:
# ── Cell 8: 국면별 섹터 ETF 성과 분석 ──────────────────────────────────────
# HMM이 분류한 국면 라벨 + 섹터 ETF 수익률 컬럼만 추출
perf_df = df[['phase'] + SECTOR_ETFS].copy()  # 국면 + 섹터 ETF 수익률만 추출

# 국면별로 그룹화하여 각 섹터 ETF의 평균 월수익률 계산 (소수→%로 변환)
phase_sector_perf = (
    perf_df.groupby('phase')[SECTOR_ETFS]  # 국면별로 그룹화하여 섹터 ETF 선택
    .___() * ___                           # Q16-1~2: 평균 집계 메소드, 퍼센트 변환 상수(100)
)
phase_sector_perf.index = [PHASE_NAMES[i] for i in phase_sector_perf.index]  # 인덱스를 국면 이름으로 변경

In [ ]:
# 1행 2열 차트 레이아웃
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── [좌측] 국면×섹터 히트맵 ──
sns.heatmap(
    phase_sector_perf,
    annot=True, fmt='.1f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    ax=axes[0],
    cbar_kws={'label': '평균 월수익률 (%)'}
)
axes[0].set_title('국면별 섹터 ETF 평균 월수익률 (%) — HMM')
axes[0].set_xlabel('섹터 ETF')
axes[0].set_ylabel('국면')

# ── [우측] 현재 국면에서 상위 3개 섹터 ──
# HMM 확률 기반으로 현재 국면 판별
all_proba = model.___(X_scaled)            # Q16-3: 각 시점별 상태 확률 예측 메소드
latest_proba_raw = all_proba[-1]           # 마지막 시점의 상태별 확률 추출
proba_by_phase = {state_to_phase[i]: float(p) for i, p in enumerate(latest_proba_raw)}  # 상태→국면 변환
current_phase_idx = ___(proba_by_phase, key=proba_by_phase.get)  # Q16-4: 최대 확률 국면 선택 함수

current_row = phase_sector_perf.loc[PHASE_NAMES[current_phase_idx]]  # 현재 국면의 섹터별 수익률
top3 = current_row.___(3)                  # Q16-5: 상위 3개 섹터 선택 메소드

colors_top = ['gold', 'silver', '#CD7F32']
axes[1].barh(
    top3.index[::-1],
    top3.values[::-1],
    color=colors_top[::-1],
    alpha=0.85
)
axes[1].set_title(f'{PHASE_NAMES[current_phase_idx]} 국면 — 상위 3 섹터')
axes[1].set_xlabel('평균 월수익률 (%)')
for i, (name, val) in enumerate(zip(top3.index[::-1], top3.values[::-1])):
    axes[1].text(val + 0.05, i, f'{val:.2f}%', va='center')

plt.tight_layout()
plt.show()

print('\n국면별 섹터 평균 월수익률 (%)')
print(phase_sector_perf.to_string(float_format='{:.2f}'.format))

## Q17. 현재 경기 국면 예측 (7 blanks)
- 확률 예측: `model.?(X_scaled)`
- 최대 확률 국면: `?(dict, key=dict.get)`
- 전이확률 접근: `model.?[state1, state2]`
- 마지막 행 접근: `df[cols].iloc[?]`


In [ ]:
# ── Cell 9: 현재 경기 국면 예측 및 추천 ────────────────────────────────────
latest_date = df.index[___]                # Q17-1: 마지막 행 인덱스 (-1)

# HMM 확률 기반 현재 국면 판별
all_proba = model.___(X_scaled)            # Q17-2: 각 시점별 상태 확률 예측 메소드
latest_proba_raw = all_proba[___]          # Q17-3: 마지막 시점의 상태별 확률 인덱스 (-1)
proba_by_phase = {state_to_phase[i]: float(p) for i, p in enumerate(latest_proba_raw)}  # 상태→국면 변환
pred_phase = ___(proba_by_phase, key=proba_by_phase.get)  # Q17-4: 최대 확률 국면 선택 함수
proba_dict = {PHASE_NAMES[ph]: round(proba_by_phase.get(ph, 0.0), 4) for ph in range(4)}  # 국면별 확률 딕셔너리

In [ ]:
# ── 예측 결과 출력 ──
print('=' * 60)
print(f'기준일: {latest_date.strftime("%Y-%m")}')
print('=' * 60)
print(f'\n  현재 국면: {PHASE_NAMES[pred_phase]}  (확률 {proba_by_phase[pred_phase]*100:.0f}%)')

# 4개 국면별 확률을 텍스트 막대로 시각화
print()
print('  국면별 확률:')
for ph in range(4):                        # 4개 국면에 대해 반복
    prob = proba_by_phase.get(ph, 0.0)     # 해당 국면의 확률 (없으면 0)
    bar = '█' * int(prob * 30)             # 확률에 비례한 막대 문자열
    print(f'  {PHASE_NAMES[ph]:12s} {prob*100:5.1f}% {bar}')

# 현재 국면에서 역사적 평균 수익률이 높았던 섹터 ETF 상위 3개
print()
print('  이 국면에서 역사적으로 강한 ETF:')
top3 = phase_sector_perf.loc[PHASE_NAMES[pred_phase]].___(___) # Q17-5~6: 상위 N개 메소드와 개수(3)
for rank, (etf, ret) in enumerate(top3.items(), 1):  # 1부터 순위 부여
    print(f'    {rank}. {etf:5s}  {ret:+.2f}% / 월')

In [ ]:
# 전이 확률 기반 다음 국면 예측
print()
print('  다음 국면 전이 확률:')
current_state = phase_order[pred_phase]    # 현재 국면에 해당하는 HMM 상태 번호
for next_ph in range(4):                   # 4개 다음 국면 후보에 대해
    next_state = phase_order[next_ph]      # 다음 국면에 해당하는 HMM 상태 번호
    trans_prob = model.___[current_state, next_state]  # Q17-7: 전이확률 행렬 속성
    if trans_prob > 0.01:                  # 1% 이상만 표시 (너무 작은 확률 제외)
        arrow = '→' if next_ph != pred_phase else '↻'  # 자기전이면 순환 화살표
        print(f'    {arrow} {PHASE_NAMES[next_ph]:12s} {trans_prob*100:5.1f}%')

# 현재 매크로 지표 수치
print()
print('  현재 매크로 지표:')
v = df[FEATURE_COLS].iloc[-1]              # 마지막 행의 피처 값 추출
print(f'  PMI: {v["pmi"]:.1f} | '
      f'금리차(10Y-3M): {v["yield_spread"]:.2f}% | '
      f'금융환경(ANFCI): {v["anfci"]:.2f}')
print(f'  실업급여 YoY: {v["icsa_yoy"]:+.1f}% | '
      f'건축허가 YoY: {v["permit_yoy"]:+.1f}% | '
      f'실질소매판매 YoY: {v["real_retail_yoy"]:+.1f}%')
print(f'  자본재주문 YoY: {v["capex_yoy"]:+.1f}% | '
      f'실질소득 YoY: {v["real_income_yoy"]:+.1f}%')
print(f'  PMI 3개월변화: {v["pmi_chg3m"]:+.2f} | '
      f'자본재주문 YoY 3개월변화: {v["capex_yoy_chg3m"]:+.2f}%p')
print('=' * 60)

## Q18. 보유 종목 국면별 성과 분석 (4 blanks)
- 그룹별 평균: `.groupby('phase')[cols].?()`
- 소수 → 퍼센트: `* ?`
- 관측 횟수: `.?()`


In [ ]:
# ── Cell 10: 보유 종목 국면별 성과 분석 ────────────────────────────────────
# HMM이 분류한 국면별 보유 종목의 평균 월수익률 계산 (소수→%로 변환)
hold_mean = df.groupby('phase')[MY_HOLDINGS].___() * ___  # Q18-1~2: 평균 메소드, 퍼센트 변환 상수(100)
hold_mean.index = [PHASE_NAMES[i] for i in hold_mean.index]  # 인덱스를 국면 이름으로 변경

In [ ]:
# 1행 2열 차트 레이아웃
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ── [좌측] 보유 종목의 국면별 수익률 히트맵 ──
if len(MY_HOLDINGS) > 1:
    sns.heatmap(
        hold_mean,
        annot=True, fmt='.2f',
        cmap='RdYlGn',
        center=0,
        linewidths=0.5,
        ax=axes[0],
        cbar_kws={'label': '평균 월수익률 (%)'}
    )
else:
    axes[0].bar(
        hold_mean.index,
        hold_mean[MY_HOLDINGS[0]],
        color=[PHASE_COLORS[i] for i in range(4)],
        alpha=0.8
    )
axes[0].set_title('보유 종목 국면별 평균 월수익률 (%) — HMM')

# ── [우측] 현재 국면에서의 보유 종목 성과 ──
current_hold = hold_mean.loc[PHASE_NAMES[pred_phase]]
bar_colors = ['#4CAF50' if v >= 0 else '#F44336' for v in current_hold.values]
axes[1].bar(
    current_hold.index,
    current_hold.values,
    color=bar_colors,
    alpha=0.85
)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_title(f'{PHASE_NAMES[pred_phase]} 국면 — 보유 종목 평균 월수익률')
axes[1].set_ylabel('%')
for i, (ticker, val) in enumerate(zip(current_hold.index, current_hold.values)):
    offset = 0.05 if val >= 0 else -0.15
    axes[1].text(i, val + offset, f'{val:+.2f}%', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# ── 텍스트 요약 ──
print('\n보유 종목 분석:')
print('-' * 50)
for phase_idx in range(4):                 # 4개 국면에 대해 반복
    phase_mask = df['phase'] == phase_idx  # 해당 국면인 행만 필터링
    n = phase_mask.sum()                   # 해당 국면의 관측 횟수
    if n == 0:                             # 관측이 없으면 건너뛰기
        continue
    is_current = ' ← 현재 국면' if phase_idx == pred_phase else ''  # 현재 국면 표시
    print(f'{PHASE_NAMES[phase_idx]} ({n}개월):{is_current}')
    for ticker in MY_HOLDINGS:             # 각 보유 종목에 대해
        avg_ret = df.loc[phase_mask, ticker].___() * ___  # Q18-3~4: 평균 메소드, 퍼센트 변환 상수
        obs_cnt = df.loc[phase_mask, ticker].count()  # 관측 횟수
        print(f'  {ticker:6s} → 평균 {avg_ret:+.2f}%  ({obs_cnt}회 관측)')
    print()

---
## 정답표

| 문제 | 빈칸 | 정답 |
|------|------|------|
| Q1-1 | HMM 클래스 | `GaussianHMM` |
| Q1-2 | 스케일러 클래스 | `StandardScaler` |
| Q2-1~4 | RANK_TO_PHASE 배열 | `3, 2, 0, 1` |
| Q3-1 | HTTP 에러 체크 | `raise_for_status` |
| Q3-2 | 인덱스 열 번호 | `0` |
| Q3-3 | 날짜 파싱 | `True` |
| Q3-4 | 숫자 변환 실패 처리 | `coerce` |
| Q3-5 | 지수 백오프 밑 | `2` |
| Q3-6 | 대기 함수 | `sleep` |
| Q4-1~2 | 변화율 메소드+기간 | `pct_change(3)` |
| Q4-3~4 | 롤링 윈도우, 최소 관측치 | `120, 36` |
| Q4-5~6 | 평균/표준편차 메소드 | `mean(), std()` |
| Q4-7~8 | PMI 스케일 변환 | `10, 50` |
| Q4-9 | 결측값 제거 | `dropna` |
| Q5-1 | 월별 평균 | `mean` |
| Q5-2~3 | YoY 변화율 | `pct_change(12)` |
| Q5-4 | 결측값 제거 | `dropna` |
| Q5-5 | 월별 평균 | `mean` |
| Q5-6 | YoY 기간 | `12` |
| Q6-1~2 | 변화율+기간 | `pct_change(12)` |
| Q6-3 | 결측값 제거 | `dropna` |
| Q6-4 | YoY 기간 | `12` |
| Q6-5 | YoY 기간 | `12` |
| Q6-6 | 결측값 제거 | `dropna` |
| Q7-1 | 병합 방식 | `outer` |
| Q7-2 | 마지막 값 | `last` |
| Q7-3 | 결측값 제거 | `dropna` |
| Q8-1 | 타임존 제거 | `None` |
| Q8-2 | 월말 값 | `last` |
| Q8-3 | 수익률 계산 | `pct_change` |
| Q9-1~2 | 차분+기간 | `diff(3)` |
| Q9-3~4 | 차분+기간 | `diff(3)` |
| Q9-5 | 병합 방식 | `left` |
| Q10-1 | 모델 평균 속성 | `means_` |
| Q10-2 | 모멘텀 가중치 | `0.5` |
| Q10-3 | 모델 평균 속성 | `means_` |
| Q10-4 | 정렬 함수 | `argsort` |
| Q11-1 | 스케일러 생성 | `StandardScaler` |
| Q11-2 | 학습+변환 | `fit_transform` |
| Q12-1~2 | 공분산 타입 | `full, diag` |
| Q12-3 | 국면 수 | `4` |
| Q12-4 | 최대 반복 횟수 | `200` |
| Q12-5 | 랜덤 시드 | `42` |
| Q12-6 | 학습 메소드 | `fit` |
| Q13-1 | 예측 메소드 | `predict` |
| Q13-2~3 | 로그우도 | `score` |
| Q14-1 | 전이확률 속성 | `transmat_` |
| Q14-2~3 | 지속기간 공식 분자/분모 | `1, 1` |
| Q14-4 | 로그우도 메소드 | `score` |
| Q14-5 | 지속기간 공식 | `1/(1-self_prob)` |
| Q15-1 | 역변환 메소드 | `inverse_transform` |
| Q15-2 | 모델 평균 속성 | `means_` |
| Q15-3 | PMI모멘텀 인덱스 | `8` |
| Q16-1 | 평균 집계 | `mean` |
| Q16-2 | 퍼센트 변환 | `100` |
| Q16-3 | 확률 예측 | `predict_proba` |
| Q16-4 | 최대값 함수 | `max` |
| Q16-5 | 상위 3개 | `nlargest` |
| Q17-1 | 마지막 인덱스 | `-1` |
| Q17-2 | 확률 예측 | `predict_proba` |
| Q17-3 | 마지막 인덱스 | `-1` |
| Q17-4 | 최대값 함수 | `max` |
| Q17-5~6 | 상위 N개 | `nlargest(3)` |
| Q17-7 | 전이확률 속성 | `transmat_` |
| Q18-1 | 평균 집계 | `mean` |
| Q18-2 | 퍼센트 변환 | `100` |
| Q18-3 | 평균 메소드 | `mean` |
| Q18-4 | 퍼센트 변환 | `100` |